# 去除寄生效应这是一个关于如何使用 scikit-rf 执行去除寄生效应操作的简短教程。我们将首先介绍去除寄生效应的概念以及为什么需要它，并提供一个简单的应用场景。接下来，我们将讨论存在哪些类型的去除寄生效应方法，以及如何为您的应用选择合适的方法。最后，我们将展示如何编写代码以快速对您的散射参数数据集执行去除寄生效应操作。## 什么是去除寄生效应以及它与校准有何不同？让我们从一个例子开始。考虑测量集成在硅晶圆上的集成射频晶体管，这对于开发用于集成电路设计的紧凑模型至关重要。在这种情况下，典型的测量设置涉及使用射频探针，该探针的一端具有同轴连接器，另一端具有接地-信号-接地 (GSG) 探针尖端。该探针尖端着陆在晶圆上构建的 GSG 焊盘上，从而可以访问被测晶体管。为了测量射频晶体管，实际的被测设备 (DUT) 通过金属互连连接到晶体管的三个端子中的两个 GSG 焊盘，同时将第三个端子连接到射频接地。在场效应晶体管的共源测量配置中，这意味着栅极和漏极端子连接到射频探针，而源极端子接地。现在，以晶体管上的端子电压为函数测量散射参数。但是，对于这种晶体管散射参数测量，校准参考平面是什么？可以通过执行标准校准（例如使用穿透-反射-传输线 (TRL) 或短路/开路/负载/直通 (SOLT) 方法）来消除电缆和探针转换的影响，从而将校准参考移动到探针尖端，或者换句话说，移动到晶圆上实现的 GSG 焊盘。使用阻抗标准衬底 (ISS) 提供一组定义明确的校准标准，可用于建立用于测量的参考平面。但是，仍然存在一个很大的“测试夹具”，它涉及金属互连，在访问我们实际想要测量的射频晶体管的实际端子之前，需要将其去除。**去除寄生效应是指去除测试夹具可能对被测设备 (DUT) 测量产生的影响。**下图（来自[此处](https://mos-ak.org/shanghai_2016/presentations/Bertrand_Ardouin_MOS-AK_Shanghai_2016.pdf））显示了一个“测试夹具”的示例，通常需要在晶圆上的测量应用中将其去除。<img src="figures/onwafer_pads.png" alt="onwafer-pads" width="500">去除寄生效应过程与校准的不同之处在于，它不使用任何已知的标准，因为它们可能不存在于某些环境中，或者在考虑到空间和成本限制的情况下，实施起来不切实际。去除寄生效应使用多个虚拟结构来帮助消除不需要的测试夹具效应，但它不能提供足够的信息来推导出完整的误差盒网络，就像使用标准校准技术获得的网络一样。由于测试夹具本身可能包含各种转换和互连，然后再到达 DUT，因此在简单的标量端口参考平面扩展不适用时，去除寄生效应很有用。有关此主题的基本介绍，请参阅[本文](http://na.support.keysight.com/faq/deembed.pdf)。

## 开路/短路去嵌入法在过去几十年中，片上射频测量中，开路/短路去嵌入法一直是射频集成电路 (RFIC) 行业的主要方法，当时晶体管的工作频率主要在几千兆赫兹的范围内。随着晶体管测量的频率范围扩大，当开路/短路去嵌入法的简化假设不再成立时，需要更复杂的方法。很难确定开路/短路去嵌入法有效性的频率上限，因为它取决于片上焊盘栅格设计中采用的布局技术。如果采用适当的设计技术，开路/短路去嵌入法至少应适用于 10 GHz，甚至更高。开路/短路去嵌入法的准确性取决于以下假设的有效性：测试夹具的寄生参数是并联电导（红色）和串联阻抗（橙色）的组合，如图所示。集总元件模型通常可以很好地代表片上测试夹具，因为首先是 GSG 焊盘中的分路焊盘电容，然后是互连线的串联阻抗。为了将测量的参考平面从校准平面（下图中的网络外部端子）移动到待测器件 (DUT)，除了 DUT 之外，还需要两个虚拟结构——开路和短路。要创建开路虚拟结构，只需将 DUT 从测试夹具中移除即可；要创建短路虚拟结构，只需将三个端子短接在一起。借助这些虚拟结构，可以提取 DUT 的实际测量结果。<img src="figures/openshort.png" alt="openshort" width="500">下图显示了开路/短路去嵌入法的详细步骤。开路虚拟结构的测量结果明显类似于并联寄生参数。只需测量开路虚拟结构（步骤 i）即可提取并联寄生参数。另一方面，短路虚拟结构同时包含并联和串联寄生参数。通过在 Y 参数域中从短路虚拟结构中减去并联寄生参数，可以提取串联寄生参数（步骤 ii）。通过分别从 Y 参数和 Z 参数域的 DUT 测量结果中减去并联和串联寄生参数，可以提取出一个不包含寄生元件的 DUT（步骤 iii）。<img src="figures/OpenShortProcedure.svg" alt="OpenShortProcedure" width="700">读者可以参考 [此详细演示文稿](https://mos-ak.org/shanghai_2016/presentations/Bertrand_Ardouin_MOS-AK_Shanghai_2016.pdf)，了解有关片上测试结构设计和最佳实践的详细信息。从现在开始，我们将重点介绍 scikit-rf 中去嵌入类的工作原理。

## 使用 Scikit-RF 进行去嵌在本节中，我们将构建一个具体的示例，以演示 Scikit-RF 中如何进行去嵌。假设被测器件是一个 1nH 电感，我们对其测量结果感兴趣。由于该电感必须放置在晶圆测试夹具中，我们假设每个端口的焊盘电容为 25fF，焊盘之间的电容为 10fF，并且每个焊盘的互连线电阻均为 2 欧姆。由此产生的网络，其测量结果在外部参考平面 P1-P2 上可用，如图所示。<img src="figures/ind_parasitics.png" alt="openshort-ckt" width="500">目标是准确提取实际的 1nH 电感，并去除所有其他不必要的寄生电路元件。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import skrf as rf

In [ ]:
# Look at the raw inductor measurement with parasitics included
# From S11/S22, it is clear that it is not a pure inductance.
raw_ind = rf.data.ind
raw_ind.plot_s_smith()

如果我们直接从这个测量结果中提取电感值，我们会得到一个随频率变化的电感值，并且质量因数也会受到寄生电阻的影响。

In [ ]:
# plot the inductance and q-factor of the raw inductor measurement
Lraw_nH = 1e9 * np.imag(1/raw_ind.y[:,0,0])/2/np.pi/raw_ind.f
Qraw = np.abs(np.imag(1/raw_ind.y[:,0,0])/np.real(1/raw_ind.y[:,0,0]))

fig, (ax1, ax2) = plt.subplots(1,2)
ax1.plot(raw_ind.f*1e-9, Lraw_nH)
ax1.grid()
ax1.set_ylabel("Inductance (nH)")
ax1.set_xlabel("Freq. (GHz)")
ax2.plot(raw_ind.f*1e-9, Qraw)
ax2.grid()
ax2.set_ylabel("Q-factor")
ax2.set_xlabel("Freq. (GHz)")
fig.tight_layout()

为了创建一个开路虚拟结构，DUT（此处为 1nH 电感）会被从 DUT 测试结构中移除，从而得到下图所示的电路。<img src="figures/ind_open.png" alt="openckt" width="500">为了创建一个短路虚拟结构，测试夹具的内部端子（这些端子原本会连接到 DUT）会被短接到地，如图所示。<img src="figures/ind_short.png" alt="shortckt" width="500">

In [ ]:
# load in 2-ports short/open dummy networks
open_nw = rf.data.open_2p
short_nw = rf.data.short_2p

为了利用可用的虚假测量数据执行开路/短路去嵌操作，我们创建一个去嵌对象，该对象是 `OpenShort` 类的实例，同时将开路和短路网络对象作为参数传递给去嵌对象的创建过程。为了获取去嵌后的网络，我们将 `deembed` 方法应用于要执行去嵌操作的网络对象。如下所示。

In [ ]:
from skrf.calibration.deembedding import OpenShort

dm = OpenShort(dummy_open=open_nw, dummy_short=short_nw, name='tutorial')

actual_ind = dm.deembed(raw_ind)
actual_ind.plot_s_smith()

In [ ]:
# plot the inductance of the de-embedded measurement
# we ignore plotting Q-factor here, because an ideal lossless inductor has infinite Q

Lactual_nH = 1e9 * np.imag(1/actual_ind.y[:,0,0])/2/np.pi/actual_ind.f

fig, ax1 = plt.subplots(1,1)
ax1.plot(actual_ind.f*1e-9, Lactual_nH)
ax1.grid()
ax1.set_ylim(0.95, 1.1)
ax1.set_ylabel("Inductance (nH)")
ax1.set_xlabel("Freq. (GHz)")
fig.tight_layout()

从上面的图可以看出，即使存在不需要的寄生元件，由于对测试夹具进行了适当的去耦处理，因此可以准确地提取出实际的电感值。

## 仅使用直通的标准进行去嵌入在高于 10 GHz 的较高频率下，开路短截线的边缘电容和短路短截线的寄生电感会导致误差。对于高频，已经提出了几种使用直通（thru）短截线的方法，例如 `SplitPi`[[1](#ref1)]、`SplitTee`[[2](#ref2)]、`AdmittanceCancel`[[3](#ref3)] 和 `ImpedanceCancel`[[4](#ref4)]。直通短截线是左侧和右侧测试焊盘的直接连接。测试设备和直通短截线的示例如图所示，其中 G 和 S 分别表示接地焊盘和信号焊盘。<img src="figures/thru_micrograph.png" alt="thru_micrograph" width="500">### 分割方法分割方法基于集总元件假设，与开路或短路方法类似。`SplitPi` 和 `SplitTee` 方法假设嵌入网络分别是 Π 网络和 T 网络，如图所示。左侧和右侧寄生元件的值（即 Z 和 Y）是从直通短截线中提取的。通过乘以左侧和右侧 ABCD 矩阵的逆矩阵来消除这些寄生元件。**这些方法仅适用于 2 端口 DUT**。<img src="figures/split_methods.svg" alt="split_methods" width="500">### 阻抗消除方法`AdmittanceCancel`（也称为 Mangan 方法）和 `ImpedanceCancel` 方法也基于集总元件假设，如图所示。但是，与分割方法不同，它们不直接计算左侧和右侧寄生元件的阻抗。这些方法通过一种称为“交换”的左右镜像运算来去嵌入 DUT。`AdmittanceCancel` 方法包括两个步骤。首先，计算一个矩阵，定义为“H”，方法是将测试设备的 ABCD 矩阵乘以直通短截线的逆矩阵，如图 (c) 所示。然后，通过取 H 的左右镜像和非镜像 Y 参数的平均值来获得去嵌入后的 Y 矩阵。类似地，在 `ImpedanceCancel` 方法中，它取 Z 参数的平均值。**这些方法仅适用于对称的 2 端口 DUT（即 S11=S22 且 S12=S21）**，但在毫米波频率下对传输线的表征方面比分割方法更准确。分割方法和阻抗消除方法之间的比较在 [[4]](https://ieeexplore.ieee.org/document/7377897) 中描述。<img src="figures/immittance_cancel.svg" alt="immittance_cancel" width="500">

## 其他去耦方法scikit-rf 中包含的其他简单的去耦方法包括 `Open`、`Short` 和 `ShortOpen` 方法，这些方法可能适用于测试夹具的寄生网络的等效电路。例如，可能希望仅从测量结果中去除焊盘接触电阻，此时可以使用 `Short` 去耦方法。在某些测量中，可能只需要去除焊盘电容，此时 `Open` 去耦方法会更合适。为了在一个操作中同时去除焊盘接触电阻和焊盘电容，`ShortOpen` 方法将是最合适的。许多其他更复杂的去耦方法已被文献报道，以提高 DUT 测量的精度，使其达到更高的频率。虽然可以使用 scikit-rf 的网络运算来实现这些方法，但将其作为软件包中内置的去耦类，将作为对该项目的开源贡献而受到欢迎。

## References<div id="ref1"></div>[1]: [L. Nan et al., “Experimental Characterization of the Effect of Metal Dummy Fills on Spiral Inductors,” in 2007 IEEE Radio Frequency Integrated Circuits (RFIC) Symposium, Jun. 2007, pp. 307–310.](https://ieeexplore.ieee.org/document/4266437)<div id="ref2"></div>[2]: [M. J. Kobrinsky et al., “Experimental validation of crosstalk simulations for on-chip interconnects using S-parameters,” IEEE Transactions on Advanced Packaging, vol. 28, no. 1, pp. 57–62, Feb. 2005.](https://ieeexplore.ieee.org/document/1391067)<div id="ref3"></div>[3]: [A. M. Mangan et al., “De-embedding transmission line measurements for accurate modeling of IC designs,” IEEE Trans. Electron Devices, vol. 53, no. 2, pp. 235–241, Feb. 2006.](https://ieeexplore.ieee.org/document/1580859)<div id="ref4"></div>[4]: [S. Amakawa et al., “Comparative analysis of on-chip transmission line de-embedding techniques,” in 2015 IEEE International Symposium on Radio-Frequency Integration Technology, Aug. 2015, pp. 91–93.](https://ieeexplore.ieee.org/document/7377897)